In [1]:
model_name = "urchade/gliner_multi-v2.1"

In [1]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
import json 
import random 


## Read JSON Data

In [192]:
train_path = "100k_aya_expanse_gliner.json"
# train_path = "data.json"

with open(train_path,"r") as f:
    data = json.load(f)

print('Dataset size:', len(data))

random.shuffle(data)
print('Dataset is shuffled...')

train_dataset = data[:int(len(data)*0.9)]
test_dataset = data[int(len(data)*0.9):]

print('Dataset is splitted...')
train_dataset[0]

Dataset size: 86116
Dataset is shuffled...
Dataset is splitted...


{'tokenized_text': ['في',
  'قطر',
  '،',
  'تعرف',
  'الداعية',
  'نورة',
  'السعدي',
  'بتمكنها',
  'من',
  'تعليم',
  'الصلاة',
  'للنساء',
  '.',
  'في',
  'درسها',
  'الديني',
  'الأسبوعي',
  '،',
  'تستشهد',
  'نورة',
  'السعدي',
  'بسورة',
  'الفاتحة',
  'لتوضيح',
  'أهمية',
  'الصلاة',
  'في',
  'الدين',
  'الإسلامي',
  '.'],
 'ner': [[4, 6, 'اسم العالم'],
  [4, 6, 'الشخصية'],
  [10, 10, 'الموضوع الديني'],
  [25, 25, 'الموضوع الديني'],
  [1, 1, 'المكان']]}

## Read JSONL data
- read and parse each line separately

In [4]:
train_path= "ner_results.jsonl"

data = []
with open(train_path,'r') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))


In [5]:
print('Dataset size:', len(data))

random.shuffle(data)
print('Dataset is shuffled...')

train_dataset = data[:int(len(data)*0.9)]
test_dataset = data[int(len(data)*0.9):]

print('Dataset is splitted...')

Dataset size: 17269
Dataset is shuffled...
Dataset is splitted...


In [6]:
train_dataset[0].keys()

dict_keys(['text', 'ner'])

In [7]:
train_dataset[0].keys()

dict_keys(['text', 'ner'])

In [8]:
train_dataset[0]

{'text': 'هو كويكب ضمن حزام الكويكبات.',
 'ner': [['1983 أر واي 2', 'كويكب'], ['32767', 'كوكب صغير']]}

In [9]:
# post processing functions

import re

def tokenize_text(text):
    """Tokenize the input text into a list of tokens."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

def extract_entities(data):
    all_examples = []

    for dt in data:

        # Attempt to extract entities; skip current record on failure
        try:
            tokens = tokenize_text(dt['text'])
            # ents = [(k["entity"], k["types"]) for k in dt['ner']]
            ents = [(x,y) for x,y in dt['ner']]
        except:
            continue

        spans = []
        for entity in ents:
            entity_tokens = tokenize_text(str(entity[0]))

            # Find the start and end indices of each entity in the tokenized text
            for i in range(len(tokens) - len(entity_tokens) + 1):
                if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                    for el in entity[1]:
                        spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ')))

        # Append the tokenized text and its corresponding named entity recognition data
        all_examples.append({"tokenized_text": tokens, "ner": spans})

    return all_examples

# generation functions
def generate_from_prompts(prompts, llm, sampling_params):
    outputs = llm.generate(prompts, sampling_params)

    all_outs = []
    
    for output in outputs:
        try:
            js = json.loads(output.outputs[0].text.strip())
        except:
            continue
            
        all_outs.append(js)

    return all_outs, extract_entities(all_outs)

In [10]:
[(x,y) for x,y in data[0]['ner']]

[('1983 أر واي 2', 'كويكب'), ('32767', 'كوكب صغير')]

In [ ]:

# all_examples = []

# for dt in data[:1]:

#     try:
#         tokens = tokenize_text(dt['text'])
#         ents = [(x,y) for x,y in dt['ner']]
#     except:
#         continue


In [ ]:
# spans = []
# for entity in ents:
#     # print(entity[0])
#     entity_tokens = tokenize_text(str(entity[0]))
#     # print(entity_tokens)
#     for i in range(len(tokens)- len(entity_tokens) + 1):
#         if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
#             print(i)
#             for el in entity:
#                 print(el)
#                 spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ')))
#     # break/


5
مجلس الوزراء
هيئة حكومية
28
المحال العامة
مؤسسة


['(', "'", 'المحال', 'العامة', "'", ',', "'", 'مؤسسة', "'", ')']

In [ ]:
# # entity_tokens = tokenize_text(str(entity[0]))

# for i in range(len(tokens) - len(entity_tokens) + 1):
#     if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
#         print(i)
#         for el in entity:
#             print(el)
#             spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ')))


In [ ]:

# # Append the tokenized text and its corresponding named entity recognition data
# all_examples.append({"tokenized_text": tokens, "ner": spans})

# all_examples


[{'tokenized_text': ['وذلك',
   'في',
   'ضوء',
   'صدور',
   'قرار',
   'مجلس',
   'الوزراء',
   'بشأن',
   'اتخاذ',
   'التدابير',
   'الاحترازية',
   'والوقائية',
   'لحماية',
   'المواطنين',
   'والحد',
   'من',
   'انتشار',
   'فيروس',
   '“',
   'كورونا',
   '”',
   'والذي',
   'نص',
   'على',
   'تنظيم',
   'مواعيد',
   'فتح',
   'وغلق',
   'المحال',
   'العامة',
   '.'],
  'ner': []},
 {'tokenized_text': ['وذلك',
   'في',
   'ضوء',
   'صدور',
   'قرار',
   'مجلس',
   'الوزراء',
   'بشأن',
   'اتخاذ',
   'التدابير',
   'الاحترازية',
   'والوقائية',
   'لحماية',
   'المواطنين',
   'والحد',
   'من',
   'انتشار',
   'فيروس',
   '“',
   'كورونا',
   '”',
   'والذي',
   'نص',
   'على',
   'تنظيم',
   'مواعيد',
   'فتح',
   'وغلق',
   'المحال',
   'العامة',
   '.'],
  'ner': []}]

In [88]:
data[0]

{'text': 'وذلك في ضوء صدور قرار مجلس الوزراء بشأن اتخاذ التدابير الاحترازية والوقائية لحماية المواطنين والحد من انتشار فيروس “كورونا” والذي نص على تنظيم مواعيد فتح وغلق المحال العامة.',
 'ner': [['مجلس الوزراء', 'هيئة حكومية'],
  ['فيروس كورونا', 'مرض'],
  ['المحال العامة', 'مؤسسة']]}

In [11]:

def extract_entities(data):
    all_examples = []

    for dt in data:
        try:
            tokens = tokenize_text(dt['text'])
            ents = [(x,y) for x,y in dt['ner']]
        except:
            continue

        spans = []
        
        for entity in ents:
            print(entity[0])
            
            entity_tokens = tokenize_text(str(entity[0]))
            print(entity_tokens)

        #     # Find the start and end indices of each entity in the tokenized text
            for i in range(len(tokens) - len(entity_tokens) + 1):
                if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                    print(i)
                    for el in entity:
                        print(el, " here")
                        spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ') if isinstance(el, str) else el))

        # # Append the tokenized text and its corresponding named entity recognition data
        all_examples.append({"tokenized_text": tokens, "ner": spans})

    return all_examples


In [12]:
def extract_entities(data):
    all_examples = []

    for dt in data:
        try:
            tokens = tokenize_text(dt['text'])
            ents = [(x,y) for x,y in dt['ner']]
        except:
            continue

        spans = []
        
        for entity in ents:
            entity_tokens = tokenize_text(str(entity[0]))
            # Find the start and end indices of each entity in the tokenized text
            for i in range(len(tokens) - len(entity_tokens) + 1):
                if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                    # Wrap the span in a list
                    spans.append([i, i + len(entity_tokens) - 1, entity[1].lower().replace('_', ' ') if isinstance(entity[1], str) else entity[1]])

        all_examples.append({"tokenized_text": tokens, "ner": spans})

    return all_examples

In [13]:
train_dataset[0]['ner']

[['1983 أر واي 2', 'كويكب'], ['32767', 'كوكب صغير']]

In [14]:
data = extract_entities(data)

In [15]:
len(data)

17237

In [242]:
data[0]
train_dataset = data[:int(len(data)*0.9)]

test_dataset = data[int(len(data)*0.9):]


In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import torch
torch.cuda.set_device('cuda:0')
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

In [2]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')


In [ ]:

model = GLiNER.from_pretrained(model_name)

In [245]:
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)


In [246]:
model.to(device)
print("done")

done


In [5]:

import torch
torch.cuda.empty_cache()
print("Available GPUs:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
# print("Trainer args:", training_args.device)

Available GPUs: 1
Current device: 0


In [248]:
num_steps = 500
batch_size =8
data_size = len(train_dataset)
num_batches = data_size // batch_size
num_epochs = max(1, num_steps // num_batches)

training_args = TrainingArguments(
    output_dir="models_unique_data",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear", #cosine
    warmup_ratio=0.1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=num_epochs,
    eval_strategy="steps",
    save_steps = 500,
    save_total_limit=10,
    dataloader_num_workers = 0,
    use_cpu = False,
    report_to="none",
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

trainer.train()

/tmp/ipykernel_357465/1731079305.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step,Training Loss,Validation Loss
500,8.027500,109.733124
1000,5.571100,99.046188
1500,5.272200,92.009888


/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/gliner/data_processing/processor.py:296: UserWarning: Sentence of length 463 has been truncated to 384
  warnings.warn(f"Sentence of length {len(tokens)} has been truncated to {max_len}")
/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/gliner/data_processing/processor.py:296: UserWarning: Sentence of length 433 has been truncated to 384
  warnings.warn(f"Sentence of length {len(tokens)} has been truncated to {max_len}")


TrainOutput(global_step=1940, training_loss=6.127728586098582, metrics={'train_runtime': 400.0164, 'train_samples_per_second': 38.781, 'train_steps_per_second': 4.85, 'total_flos': 0.0, 'train_loss': 6.127728586098582, 'epoch': 1.0})

In [3]:
# trained_model = GLiNER.from_pretrained("models_unique_data/checkpoint-1940", load_tokenizer=True)
# trained_model = GLiNER.from_pretrained("models/checkpoint-2209", load_tokenizer=True)
trained_model = GLiNER.from_pretrained("NAMAA-Space/gliner_arabic-v2.1_rich",load_tokenizer=True)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [5]:
domains = {
    "General": {
        "texts": [
            """
            فاز ليونيل ميسي بجائزة الكرة الذهبية لعام 2023 مع نادي باريس سان جيرمان بعد قيادته الأرجنتين في كأس العالم.
            """,
            """
            حصل نجيب محفوظ على جائزة نوبل للآداب عام 1988 عن روايته "أولاد حارتنا" في احتفال بالقاهرة.
            """
        ],
        "labels": ["شخص", "جائزة", "منظمة", "موقع", "حدث"]
    },
    "Medical": {
        "texts": [
            """
            اكتشف الدكتور محمد صلاح علاجًا جديدًا لمرض السكري في مستشفى الملك فيصل بالرياض باستخدام الذكاء الاصطناعي.
            """,
            """
            نجحت عملية زراعة قلب في مستشفى الحرس الوطني بجدة بقيادة الدكتورة فاطمة الزهراني.
            """
        ],
        "labels": ["شخص", "مرض", "علاج", "مستشفى", "إجراء طبي"]
    },
    "Regional": {
        "texts": [
            """
            افتتح أمير مكة المكرمة خالد الفيصل مشروع توسعة الحرم المكي في المملكة العربية السعودية.
            """,
            """
            أعلن الشيخ محمد بن راشد عن إطلاق مهرجان دبي للتسوق 2025 في إمارة دبي.
            """
        ],
        "labels": ["شخص", "منطقة", "مشروع", "بلد", "حدث"]
    },
    "Financial": {
        "texts": [
            """
            قفز سهم شركة سابك بنسبة 7% في سوق تداول السعودي بعد إعلان أرباح الربع الأول لعام 2025.
            """,
            """
            أصدر مصرف الإمارات المركزي عملة جديدة بقيمة 1000 درهم لتعزيز الاقتصاد الوطني.
            """
        ],
        "labels": ["منظمة", "سهم", "سوق", "أداة مالية", "عملة"]
    },
    "Historical": {
        "texts": [
            """
            قاد صلاح الدين الأيوبي معركة حطين عام 1187 لاستعادة القدس من الصليبيين.
            """,
            """
            شهدت ثورة 1919 في مصر قيادة سعد زغلول ضد الاحتلال البريطاني في القاهرة.
            """
        ],
        "labels": ["شخص", "معركة", "موقع", "تاريخ", "حدث"]
    },
    "Religious": {
        "texts": [
            """
            أمر الخليفة عمر بن الخطاب ببناء المسجد الأقصى في القدس خلال الفتح الإسلامي.
            """,
            """
            يزور ملايين المسلمين الكعبة المشرفة في مكة المكرمة لحج 1446 هـ.
            """
        ],
        "labels": ["شخص", "موقع ديني", "حدث ديني", "تاريخ", "جماعة"]
    },
    "Educational": {
        "texts": [
            """
            افتتحت جامعة الملك سعود فرعًا جديدًا في الخرج بقيادة الدكتور عبد الرحمن العثيمين.
            """,
            """
            حصلت طالبة ليلى أحمد على المركز الأول في مسابقة القرآن الكريم بجامعة الأزهر.
            """
        ],
        "labels": ["منظمة تعليمية", "شخص", "موقع", "مسابقة", "جائزة"]
    },
    "Technological": {
        "texts": [
            """
            أطلقت شركة stc السعودية شبكة 6G في الرياض بإشراف المهندس خالد البابطين.
            """,
            """
            طورت شركة هواوي العربية جهازًا ذكيًا جديدًا في مركز أبحاث دبي.
            """
        ],
        "labels": ["منظمة", "تقنية", "موقع", "شخص", "منتج"]
    },
    "Cultural": {
        "texts": [
            """
            أحيا الفنان محمد منير حفلًا غنائيًا في مهرجان جرش الثقافي بالأردن.
            """,
            """
            عرضت الفنانة منى زكي مسرحية "حلم ليلة صيف" في دار الأوبرا المصرية بالقاهرة.
            """
        ],
        "labels": ["شخص", "حدث ثقافي", "موقع", "عمل فني", "منظمة"]
    },
    "Agricultural": {
        "texts": [
            """
            زرع المزارع أحمد الشمري محصول القمح في واحة الأحساء بالمنطقة الشرقية.
            """,
            """
            أطلقت وزارة الزراعة السعودية مشروع ري حديث في تبوك لدعم إنتاج التمور.
            """
        ],
        "labels": ["شخص", "محصول", "موقع", "مشروع", "منظمة"]
    },
    "Tourism": {
        "texts": [
            """
            افتتحت شركة العلا للسياحة موقعًا أثريًا جديدًا في مدينة الحجر بالسعودية.
            """,
            """
            استقبلت شرم الشيخ أكثر من مليون سائح في موسم الصيف 2025.
            """
        ],
        "labels": ["منظمة", "موقع سياحي", "حدث", "منطقة", "جماعة"]
    }
}

# Test the model across different domains
for domain, data in domains.items():
    print(f"\n=== اختبار المجال: {domain} ===")
    texts = data["texts"]
    labels = data["labels"]
    
    for i, text in enumerate(texts, 1):
        print(f"\nاختبار النص {i} في مجال {domain}:")
        # Predict entities with a confidence threshold of 0.5
        entities = trained_model.predict_entities(text.strip(), labels, threshold=0.5)
        
        # Print detected entities
        if entities:
            for entity in entities:
                print(f"{entity['text']} => {entity['label']} (confidence: {entity['score']:.3f})")
        else:
            print("لم يتم العثور على كيانات في هذا النص")

# Final summary
print("\nتم الانتهاء من اختبار جميع المجالات!")


=== اختبار المجال: General ===

اختبار النص 1 في مجال General:
ليونيل ميسي => شخص (confidence: 0.936)
نادي باريس سان جيرمان => منظمة (confidence: 0.733)
الأرجنتين => موقع (confidence: 0.570)
كأس العالم => حدث (confidence: 0.774)

اختبار النص 2 في مجال General:
نجيب محفوظ => شخص (confidence: 0.950)
جائزة نوبل للآداب => جائزة (confidence: 0.765)

=== اختبار المجال: Medical ===

اختبار النص 1 في مجال Medical:
الدكتور محمد صلاح => شخص (confidence: 0.902)
السكري => مرض (confidence: 0.686)
مستشفى الملك فيصل => مستشفى (confidence: 0.908)
الذكاء الاصطناعي => علاج (confidence: 0.631)

اختبار النص 2 في مجال Medical:
عملية زراعة قلب => إجراء طبي (confidence: 0.664)
مستشفى الحرس الوطني بجدة => مستشفى (confidence: 0.893)
الدكتورة فاطمة الزهراني => شخص (confidence: 0.849)

=== اختبار المجال: Regional ===

اختبار النص 1 في مجال Regional:
خالد الفيصل => شخص (confidence: 0.852)
مشروع توسعة الحرم المكي => مشروع (confidence: 0.689)
المملكة العربية السعودية => بلد (confidence: 0.875)

اختبار النص 2 في مج

In [6]:
text = "غزة، مدينة يصمد شعبها الفلسطيني المحاصر بقلوب كالصخر، يواجهون الإبادة الجماعية من الكيان الصهيوني برعاية أمريكية وخذلان العالم أجمع، حيث يقاوم أهلها، بقيادة يحيى السنوار ومحمد الضيف، مع فصائل حماس تحت القصف والحصار والموت منذ 7 أكتوبر 2023، وسط صمت الأمم المتحدة والاتحاد الأوروبي، بينما تجري مفاوضات في القاهرة بوساطة مصر وقطر."

labels = ["person", "organization", "location", "date"]
entities = trained_model.predict_entities(text, labels)

for entity in entities:
    print(f"Entity: {entity['text']} | Label: {entity['label']} | Score: {entity['score']:.3f}")


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Entity: غزة | Label: location | Score: 0.833
Entity: الكيان الصهيوني | Label: organization | Score: 0.819
Entity: يحيى السنوار | Label: person | Score: 0.916
Entity: فصائل حماس | Label: organization | Score: 0.686
Entity: 7 أكتوبر 2023 | Label: date | Score: 0.854
Entity: الأمم المتحدة | Label: organization | Score: 0.829
Entity: القاهرة | Label: location | Score: 0.798
Entity: مصر | Label: location | Score: 0.695


In [ ]:
# trained_model.push_to_hub("gliner_arabic-v2.1_rich")

pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Abdelkareem/gliner_arabic-v2.1_rich/commit/1d5faaf58f14d13518fcebaee22a215d69e44afd', commit_message='Push model using huggingface_hub.', commit_description='', oid='1d5faaf58f14d13518fcebaee22a215d69e44afd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Abdelkareem/gliner_arabic-v2.1_rich', endpoint='https://huggingface.co', repo_type='model', repo_id='Abdelkareem/gliner_arabic-v2.1_rich'), pr_revision=None, pr_num=None)

In [ ]:
from gliner import GLiNER

Entity: محمد بن زايد | Label: person | Score: 0.804
Entity: الملك عبدالله الثاني | Label: person | Score: 0.758
Entity: أبوظبي | Label: location | Score: 0.861
Entity: ١٥ أكتوبر ٢٠٢٤ | Label: date | Score: 0.852


In [ ]:
print(train_dataset[0])  # Show the first record

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Convert train_dataset to a DataFrame
train_df = pd.DataFrame(train_dataset)

# Convert test_dataset to a DataFrame (assuming it has the same structure)
test_df = pd.DataFrame(test_dataset)

# Inspect the DataFrame
print(train_df.head())

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value
from huggingface_hub import login

import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value



In [ ]:
print(train_df.iloc[0])

In [ ]:
import pandas as pd
from datasets import Dataset


def convert_ner_to_dict(ner_list):
    return [
        {"start": entity[0], "end": entity[1], "label": entity[2]}
        for entity in ner_list
    ]


train_df["ner"] = train_df["ner"].apply(convert_ner_to_dict)


In [ ]:

test_df["ner"] = test_df["ner"].apply(convert_ner_to_dict)

# Convert the DataFrame to a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Push to Hugging Face Hub
# train_dataset.push_to_hub("100k_aya_expanse_ner_gliner")
hf = DatasetDict({"train": train_dataset, "test_dataset": test_dataset})

In [ ]:
hf.push_to_hub("100k_aya_expanse_ner_gliner")